# 06 — Ingestion réelle & inspection (poste de prod)

Outil **interactif** et **non bloquant** pour, sur le poste de production :

1. **Inventorier** les vrais fichiers de `data/` (OECI par année + régional `canceroBR`).
2. **Inspecter en détail** le format brut de chaque source (en-têtes réels, modalités
   observées vs attendues au YAML) — pour *diagnostiquer* un fichier non conforme.
3. **Fabriquer** le format long `data/donnees.csv` via le pipeline (`chargeur_long`), en
   voyant le journal d'ingestion et un compte-rendu par source.
4. **Inspecter** le format long produit (couverture, disponibilité de la survie par année,
   contrôle de contrat).
5. **Explorer** librement.

> Complément du notebook **05** (garde-fou binaire qui *bloque* le build). Ici, rien ne bloque :
> on affiche et on diagnostique. Réf. : `docs/contrat_donnees_pivot.md`,
> `docs/descriptif_sources.yaml`.

In [ ]:
# Préambule : imports, chemins, vocabulaire du contrat.
import pandas as pd, numpy as np, yaml
import glob, os, re, io, contextlib
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 200)

DATA_DIR   = Path("../data")
YAML_PATH  = "../docs/descriptif_sources.yaml"
CHEMIN_CSV = DATA_DIR / "donnees.csv"

# Période attendue (cf. contrat / referentiels.ANNEE_MIN..MAX)
ANNEE_MIN, ANNEE_MAX = 2022, 2025
ANNEES_ATTENDUES = list(range(ANNEE_MIN, ANNEE_MAX + 1))

# Descriptif des sources = vérité du format BRUT des fichiers réels.
DESC = yaml.safe_load(open(YAML_PATH, encoding="utf-8"))
ATT  = DESC.get("modalites_attendues", {})

# Vocabulaire du contrat (pour l'inspection du format long, section 4).
COLONNES = ["annee","source","niveau","entite","appareil","organe",
            "age","stade","population","variable","valeur"]
DOMAINES = {
    "source":     {"BN","DIM APHP","EDS APHP"},
    "niveau":     {"aphp","ghu","hopital","type_etab"},
    "age":        {"tous","pédiatrie","adultes"},
    "population": {"tous","nouveaux"},
    "stade":      {"I-III","IV"},
}
VARIABLES = {"nb_patients","nb_sejours_chirurgie","nb_sejours_chimiotherapie",
             "nb_sejours_radiotherapie","nb_sejours_palliatifs",
             "delai_global_median","delai_chirurgie_median",
             "delai_traitement_medical_median","delai_radio_median",
             "nb_patients_stade","survie_1an","survie_5ans"}
SURVIE_VARS = {"nb_patients_stade","survie_1an","survie_5ans"}

def _kv(n):
    for u in ["o","Ko","Mo","Go"]:
        if n < 1024: return f"{n:.0f} {u}"
        n /= 1024
    return f"{n:.0f} To"

def _portee_aphp(niveau):
    """Niveau canceroAPHP « portée - granularité » → True si la PORTÉE est AP-HP.
    Robuste aux libellés bancals : supprime TOUS les espaces puis rsplit sur le DERNIER
    '-' (la portée 'AP-HP' conserve son tiret interne). Faux pour GH/GHU/Hop."""
    norm = re.sub(r"\s+", "", str(niveau))          # supprime TOUS les espaces
    portee = norm.rsplit("-", 1)[0] if "-" in norm else norm
    return portee.upper() == "AP-HP"                 # True pour AP-HP-*, faux pour GH/GHU/Hop

def _portee_de(niveau):
    """Portée normalisée (majuscules, sans espaces) d'un libellé « portée - granularité »."""
    norm = re.sub(r"\s+", "", str(niveau))
    return (norm.rsplit("-", 1)[0] if "-" in norm else norm).upper()

print("Descriptif chargé — sources :", list(DESC.get("sources", {}).keys()),
      "| période attendue :", ANNEES_ATTENDUES)

## 1. Inventaire des fichiers réels

In [ ]:
def _annee_oeci(nom):
    m = re.search(r"indicateurs_oeci_(\d{4})", nom)
    return int(m.group(1)) if m else None

def _infos(path):
    st = os.stat(path)
    return {"fichier": os.path.basename(path), "taille": _kv(st.st_size),
            "modifié": pd.Timestamp(st.st_mtime, unit="s").strftime("%Y-%m-%d %H:%M")}

# OECI — un fichier par année (on exclut les gabarits *_fictif)
oeci = sorted(p for p in glob.glob(str(DATA_DIR / "indicateurs_oeci_*.xlsx"))
              if "_fictif" not in os.path.basename(p))
rows = []
for p in oeci:
    d = _infos(p); d["année"] = _annee_oeci(d["fichier"]); rows.append(d)
oeci_df = pd.DataFrame(rows, columns=["année","fichier","taille","modifié"])
if not oeci_df.empty: oeci_df = oeci_df.sort_values("année")
print(f"OECI — {len(oeci_df)} fichier(s)")
display(oeci_df)

annees_presentes = set(int(a) for a in oeci_df["année"].dropna())
manquantes = [a for a in ANNEES_ATTENDUES if a not in annees_presentes]
print("Années OECI manquantes vs période attendue :", manquantes if manquantes else "aucune ✓")

# Régional — TOUTES les sources non-OECI du YAML (canceroBR = comparateurs ;
# canceroAPHP = entité AP-HP dédupliquée). Patients + Séjours, multi-années. Absent → info.
reg_rows, n_present = [], 0
for src_cle, src_conf in DESC["sources"].items():
    if src_cle == "oeci" or "fichiers" not in src_conf:
        continue
    for role, motif in src_conf["fichiers"].items():
        found = [q for q in sorted(glob.glob(str(DATA_DIR / motif)))
                 if "_fictif" not in os.path.basename(q)]
        for q in found:
            d = _infos(q); d["source"] = src_cle; d["rôle"] = role; reg_rows.append(d); n_present += 1
        if not found:
            reg_rows.append({"source": src_cle, "rôle": role,
                             "fichier": f"(absent : {motif})", "taille": "—", "modifié": "—"})
reg_df = pd.DataFrame(reg_rows, columns=["source","rôle","fichier","taille","modifié"])
print(f"\nRégional (canceroBR + canceroAPHP) — {n_present} fichier(s) présent(s)")
display(reg_df if not reg_df.empty else "aucune source régionale déclarée au YAML")

## 2. Inspection détaillée des sources brutes

Pour chaque feuille déclarée au YAML : en-têtes réellement lus, modalités **observées** de
`Niveau` vs déclarées, présence des colonnes de mesures, et échantillon des valeurs GHU/Statut.
On affiche le **détail** (pas un simple ✓/✗) pour pouvoir corriger la source si besoin.
Statuts : ✓ conforme · ⚠ modalité nouvelle/à vérifier · ✗ mesure/feuille attendue absente · ○ info.

In [ ]:
# Lecture BRUTE via pandas (header=None) : une seule lecture par feuille,
# on évite l'accès aléatoire openpyxl read_only (piège quadratique).
def _lire_brut(path, sheet):
    return pd.read_excel(path, sheet_name=sheet, header=None, dtype=object)

def _propre(v):
    s = "" if v is None else str(v).strip()
    return "" if s.lower() == "nan" else s

def _entetes(raw, n_entete):
    libs = []
    for r in range(min(n_entete, len(raw))):
        libs += [x for x in (_propre(v) for v in raw.iloc[r].tolist()) if x]
    return libs

def _col_vals(raw, col_1based, first_data_row):
    idx = col_1based - 1
    if idx >= raw.shape[1]: return []
    s = raw.iloc[first_data_row - 1:, idx]
    return [x for x in (_propre(v) for v in s.tolist()) if x]

def _inspecter(raw, conf, ctx, rap, portee_retenue=None):
    ent = _entetes(raw, conf.get("lignes_entete", 1)); ent_txt = " | ".join(ent)
    dims = conf.get("dimensions", {}); conf_niv = conf.get("niveau", {})
    fdr = conf.get("premiere_ligne_donnees", 2)
    if portee_retenue:
        # Source canceroAPHP : Niveau = « portée - granularité ». On ne retient que la
        # portée AP-HP ; GH/GHU/Hop sont IGNORÉS volontairement (info ○, pas une anomalie).
        obs = sorted(set(_col_vals(raw, dims["niveau"], fdr)))
        portees = sorted(set(_portee_de(v) for v in obs))
        aphp = [v for v in obs if _portee_aphp(v)]
        ignorees = [p for p in portees if p != "AP-HP"]
        if not obs:
            rap.append({**ctx, "contrôle": "Portée (AP-HP retenue)", "statut": "○",
                        "détail": "aucune modalité Niveau lue (fichier vide ?)"})
        else:
            rap.append({**ctx, "contrôle": "Portée (AP-HP retenue)",
                        "statut": "✓" if aphp else "✗",
                        "détail": (f"portées observées={portees} ; AP-HP "
                                   + ("présente" if aphp else "ABSENTE (fichier peuplé)"))})
        if ignorees:
            rap.append({**ctx, "contrôle": "Portées ignorées (loader)", "statut": "○",
                        "détail": f"{ignorees} — ignorées volontairement (hors AP-HP)"})
    elif "niveau" in dims:
        obs = sorted(set(_col_vals(raw, dims["niveau"], fdr)))
        if isinstance(conf_niv, dict) and conf_niv.get("mode") == "mots_cles":
            mots = conf_niv.get("mots_cles", {})
            non_rec = [m for m in obs if not any(k.lower() in m.lower() for k in mots)]
            rap.append({**ctx, "contrôle": "Niveau (mots-clés)",
                        "statut": "✓" if not non_rec else "⚠",
                        "détail": f"{len(obs)} modalité(s) ; non reconnues={non_rec or '∅'}"})
        else:
            decl = set(conf_niv.keys())
            nouv = sorted(set(obs) - decl); absentes = sorted(decl - set(obs))
            rap.append({**ctx, "contrôle": "Niveau",
                        "statut": "✓" if not nouv else "⚠",
                        "détail": f"nouvelles={nouv or '∅'} ; déclarées-absentes={absentes or '∅'}"})
    # Colonnes de mesures présentes dans l'en-tête ? (pour Sej, doublon toléré : présence suffit)
    mes = conf.get("mesures", {}); disp = mes.get("disposition")
    attendues = mes.get("colonnes_utiles", []) if disp == "simple" \
        else list(mes.get("mapping_blocs", {}).keys()) if disp == "blocs" else []
    if attendues:
        manq = [c for c in attendues if c not in ent_txt]
        rap.append({**ctx, "contrôle": f"Mesures ({disp})",
                    "statut": "✓" if not manq else "✗",
                    "détail": f"absentes de l'en-tête={manq or '∅'}"})
    elif disp == "plan_survie":
        rap.append({**ctx, "contrôle": "Mesures (plan_survie)", "statut": "○",
                    "détail": "structure population×horizon×stade (résolue à l'ingestion)"})
    # GHU brut (échantillon)
    if "ghu" in dims:
        gobs = sorted(set(_col_vals(raw, dims["ghu"], fdr)))[:8]
        rap.append({**ctx, "contrôle": f"GHU (forme {conf.get('ghu_forme','?')})",
                    "statut": "○", "détail": f"observé (échantillon)={gobs or '∅'}"})
    # Statut régional brut (échantillon)
    if "statut" in dims:
        sobs = sorted(set(_col_vals(raw, dims["statut"], fdr)))[:12]
        rap.append({**ctx, "contrôle": "Statut (observé)", "statut": "○", "détail": f"{sobs or '∅'}"})
    return ent_txt

rapport, entetes_dump = [], []

# OECI : toutes les années présentes
oeci_conf = DESC["sources"]["oeci"]["feuilles"]
for p in oeci:
    annee = _annee_oeci(os.path.basename(p))
    try:
        xls = pd.ExcelFile(p); feuilles = set(xls.sheet_names)
    except Exception as e:
        rapport.append({"source": f"OECI {annee}", "feuille": "—",
                        "contrôle": "ouverture", "statut": "✗", "détail": f"{type(e).__name__}: {e}"}); continue
    for cle, conf in oeci_conf.items():
        nom = conf["nom"]; ctx = {"source": f"OECI {annee}", "feuille": nom}
        if nom not in feuilles:
            rapport.append({**ctx, "contrôle": "présence feuille",
                            "statut": "✗" if conf.get("obligatoire") else "○", "détail": "absente"}); continue
        raw = _lire_brut(p, nom)
        ent = _inspecter(raw, conf, ctx, rapport)
        entetes_dump.append((f"OECI {annee}", nom, ent))

# Sources non-OECI (régional canceroBR + AP-HP dédupliqué canceroAPHP) : itère TOUTE source
# ayant « fichiers » + « feuilles ». Une source portant « portee_retenue » (canceroAPHP) est
# inspectée par PORTÉE (GH/GHU/Hop ignorés = info, pas anomalie).
for src_cle, src_conf in DESC["sources"].items():
    if src_cle == "oeci" or "fichiers" not in src_conf or "feuilles" not in src_conf:
        continue
    fics = src_conf.get("fichiers", {}); portee = src_conf.get("portee_retenue")
    for cle, conf in src_conf["feuilles"].items():
        role = conf["fichier"]; motif = fics.get(role, "")
        paths = [q for q in glob.glob(str(DATA_DIR / motif)) if "_fictif" not in os.path.basename(q)]
        ctx = {"source": f"{src_cle} ({role})", "feuille": conf["nom"]}
        if not paths:
            rapport.append({**ctx, "contrôle": "présence fichier", "statut": "○", "détail": f"aucun {motif}"}); continue
        try:
            xls = pd.ExcelFile(paths[0])
            if conf["nom"] not in xls.sheet_names:
                rapport.append({**ctx, "contrôle": "présence feuille", "statut": "✗", "détail": "absente"}); continue
            raw = _lire_brut(paths[0], conf["nom"])
            ent = _inspecter(raw, conf, ctx, rapport, portee_retenue=portee)
            entetes_dump.append((f"{src_cle} ({role})", conf["nom"], ent))
        except Exception as e:
            rapport.append({**ctx, "contrôle": "ouverture", "statut": "✗", "détail": f"{type(e).__name__}: {e}"})

rap_df = pd.DataFrame(rapport, columns=["source","feuille","contrôle","statut","détail"])
print(f"Rapport d'inspection — {len(rap_df)} ligne(s)\n")
display(rap_df)

anomalies = rap_df[rap_df["statut"].isin(["✗","⚠"])]
print("\n⚠ Anomalies (✗/⚠) :", "aucune ✓" if anomalies.empty else f"{len(anomalies)} — voir ci-dessous")
if not anomalies.empty: display(anomalies)

print("\n— En-têtes réellement lus (échantillon) —")
for src, feuille, ent in entetes_dump:
    print(f"\n[{src}] « {feuille} »")
    print("   " + (ent[:300] + (" …" if len(ent) > 300 else "")))

## 3. Génération du format long (`chargeur_long`)

On appelle le **pipeline réel** (pas de logique dupliquée). Le journal capture l'ingestion, les
avertissements de **dérive** (`⚠ Dérive …`) et l'alerte régional-vide, puis un compte-rendu par source.

In [ ]:
import sys
sys.path.insert(0, "../src")            # SEULE zone qui importe le pipeline
from export_internes import exporter_csv

buf, ok_gen, err = io.StringIO(), True, ""
try:
    with contextlib.redirect_stdout(buf):
        exporter_csv(dossier_data=str(DATA_DIR), fictif=False, dossier_source=str(DATA_DIR))
except Exception as e:
    ok_gen, err = False, f"{type(e).__name__}: {e}"

print("— Journal d'ingestion (chargeur_long) —")
print(buf.getvalue().rstrip() or "(aucune sortie)")
if not ok_gen:
    print("\n✗ Ingestion échouée :", err)
    print("  → vérifier l'inventaire (§1) et les anomalies (§2) : fichier/feuille manquant ou format dérivé.")
else:
    st = os.stat(CHEMIN_CSV)
    print(f"\n✓ donnees.csv (re)généré : {_kv(st.st_size)} · {CHEMIN_CSV}")
    dg = pd.read_csv(CHEMIN_CSV, dtype={"stade": "string"})
    print("\n— Compte-rendu par source —")
    display(dg.groupby("source").agg(lignes=("valeur", "size"),
                                     variables=("variable", "nunique"),
                                     niveaux=("niveau", "nunique"),
                                     années=("annee", "nunique")).reset_index())
    print("\n— Lignes par (source × niveau) —");  display(pd.crosstab(dg["source"], dg["niveau"]))
    print("\n— Lignes par (source × année) —");   display(pd.crosstab(dg["source"], dg["annee"]))

## 4. Inspection du format long produit

Couverture, **disponibilité de la survie par année** (le trou 2023/2024 apparaît ici si les
extraits sont partiels) et contrôle de contrat détaillé — informatif, non bloquant.

In [ ]:
try:
    d = pd.read_csv(CHEMIN_CSV, dtype={"stade": "string"})
except FileNotFoundError:
    d = None
    print("Pas de donnees.csv — lancer la section 3 d'abord.")

if d is not None:
    print("shape :", d.shape)
    display(d.head(8))

    print("\n— Couverture : variable × (source, niveau) —")
    display(pd.crosstab(d["variable"], [d["source"], d["niveau"]]))

    print("\n— Entité AP-HP du régional (résultat attendu de la dédup canceroAPHP) —")
    bn_aphp = d[(d["source"] == "BN") & (d["niveau"] == "aphp") & (d["variable"] == "nb_patients")]
    if bn_aphp.empty:
        print("  ✗ AUCUNE ligne BN / niveau aphp / nb_patients — l'entité AP-HP régionale est ABSENTE "
              "(canceroAPHP non ingéré ?).")
    else:
        nz = pd.to_numeric(bn_aphp["valeur"], errors="coerce").fillna(0)
        annees = sorted(int(a) for a in bn_aphp["annee"].unique())
        peuple = bool((nz > 0).any())
        print(f"  {'✓' if peuple else '○'} BN/aphp/nb_patients présent : {len(bn_aphp)} ligne(s), "
              f"années {annees}"
              + ("" if peuple else " — valeurs toutes nulles (fichiers canceroAPHP vides ?)"))

    print("\n— Disponibilité de la SURVIE par année —")
    surv = d[d["variable"].isin(SURVIE_VARS)]
    if surv.empty:
        print("  aucune ligne de survie.")
    else:
        display(pd.crosstab(surv["variable"], surv["annee"]))
        annees_surv = sorted(int(a) for a in surv["annee"].unique())
        trous = [a for a in ANNEES_ATTENDUES if a not in annees_surv]
        print("  survie présente :", annees_surv, "| manquante vs période :", trous or "aucune")

    print("\n— Entités distinctes par niveau —")
    display(d.groupby("niveau")["entite"].nunique().rename("nb_entités").reset_index())
    print("NB : les valeurs masquées/absentes (« — ») ne sont PAS des NaN ici — le chargeur "
          "n'émet pas les valeurs absentes → elles se lisent comme des LIGNES MANQUANTES.")

    print("\n— Contrôle de contrat (informatif, non bloquant) —")
    ecarts = []
    manq = set(COLONNES) - set(d.columns); sup = set(d.columns) - set(COLONNES)
    if manq or sup: ecarts.append(f"colonnes : manquantes={sorted(manq) or '∅'}, en trop={sorted(sup) or '∅'}")
    for col, dom in DOMAINES.items():
        if col in d.columns:
            hors = set(d[col].dropna().unique()) - dom
            if hors: ecarts.append(f"{col} hors domaine : {sorted(hors)}")
    hv = set(d["variable"].dropna().unique()) - VARIABLES
    if hv: ecarts.append(f"variables inconnues : {sorted(hv)}")
    cle = [c for c in COLONNES if c != "valeur"]
    ndup = int(d.duplicated(subset=[c for c in cle if c in d.columns]).sum())
    if ndup: ecarts.append(f"{ndup} doublon(s) de clé")
    if "valeur" in d.columns:
        nn = d["valeur"].notna() & pd.to_numeric(d["valeur"], errors="coerce").isna()
        if int(nn.sum()): ecarts.append(f"{int(nn.sum())} valeur(s) non numérique(s)")
    if "annee" in d.columns:
        an = pd.to_numeric(d["annee"], errors="coerce")
        if not an.between(ANNEE_MIN, ANNEE_MAX).all():
            ecarts.append(f"années hors [{ANNEE_MIN},{ANNEE_MAX}] : {sorted(set(an.dropna().astype(int)))}")
    if ecarts:
        for e in ecarts: print("  ✗", e)
    else:
        print("  ✓ conforme au contrat (colonnes, domaines, vocabulaire, clé, valeurs, période).")

## 5. Exploration libre

In [ ]:
d = pd.read_csv(CHEMIN_CSV, dtype={"stade": "string"})

# Ex. 1 — délais médians par GHU (appareil TOTAL) :
# q = d[(d.source=="DIM APHP") & (d.niveau=="ghu") & d.variable.str.startswith("delai_") & (d.appareil=="TOTAL")]
# display(q.pivot_table(index="entite", columns=["variable","annee"], values="valeur"))

# Ex. 2 — survie 5 ans AP-HP par appareil (stade I-III) et par année :
# q = d[(d.variable=="survie_5ans") & (d.stade=="I-III") & (d.niveau=="aphp") & (d.organe=="TOTAL")]
# display(q.pivot_table(index="appareil", columns="annee", values="valeur"))

# Ex. 3 — patients (tous vs nouveaux) AP-HP par année :
# q = d[(d.variable=="nb_patients") & (d.niveau=="aphp") & (d.appareil=="TOTAL") & (d.organe=="TOTAL")]
# display(q.pivot_table(index="population", columns="annee", values="valeur"))

print("Prêt — dé-commenter un exemple ou écrire vos propres requêtes sur `d`.")